1/x in encoder1 4670.44, 181 degrees
1/x in encoder2 68336.18, 326 degrees

Conclusion
6514.89 -> introduce a lot of error

In [1]:
import subprocess
from datasets import load_dataset
import re
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

dataset = load_dataset("dair-ai/emotion", split="test")

/Users/tonyma/code/FHE-BERT-Tiny-Emotion/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gokuls/BERT-tiny-emotion-intent")

texts = dataset['text']
token_length = [len(tokenizer.encode(entry['text'], truncation=False)) for entry in dataset]
labels = dataset['label']

df = pd.DataFrame({
    'text': texts,
    'token_length': token_length,
    'label': labels
})

# sort label
df_label = df.sort_values(by=['label', 'token_length'], ascending=[True, True]).reset_index(drop=True)

In [3]:
new_df2 = df_label.groupby('label').head(10).reset_index(drop=True)
print(len(new_df2))
print(new_df2[:21])

60
                            text  token_length  label
0          i feel so embarrassed             6      0
1             i do feel stressed             6      0
2                i feels so lame             6      0
3      i feel embarrassed enough             6      0
4          i stop feeling guilty             6      0
5         i feel depressed again             6      0
6              i feel soo lonely             6      0
7     im feeling depressed again             6      0
8           i just feel troubled             6      0
9              i feel less alone             6      0
10          im feeling energetic             5      1
11          i feel more creative             6      1
12        i feel really inspired             6      1
13         i feel more energetic             6      1
14             i feel any better             6      1
15       i would feel productive             6      1
16   im feeling hopeful relieved             6      1
17           i feel gorge

In [4]:
correct = []
error = []
output = []
execution_times = []

func_error = []
count = 0

def run_fhe_bert_tiny(sample):
    global correct, error, func_error, output, execution_times, count
    #print(sample)
    
    sentence = sample.text
    print("sentence: ", sentence)
    
    cmd = [f"./FHE-BERT-Tiny", sentence]
    #cmd = ["./FHE-BERT-Tiny", sentence, "--verbose"]
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )
    
    # format
    result = result.stdout.strip()
    #print("raw result", result)
    
    # check [x, x, x, x, x, x]
    #pattern = r'\[(.*?)\]'
    pattern = r'plain_result: \[(.*?)\]'
    match = re.search(pattern, result)
    tmp = match.group(1).split(",")
    res = [float(x) for x in tmp]
    output.append(res)
    print(res)
    
    # execution time
    # The evaluation of the FHE circuit took: 121 seconds.
    pattern_time = r'The evaluation of the FHE circuit took:\s*(\d+)\s*seconds'
    match_time = re.search(pattern_time, result)
    seconds = float(match_time.group(1))
    execution_times.append(seconds)
        
    if ("Outcome: sadness" in result):
        if (sample.label == 0):
            correct.append(count)
            print("sadness")
        else:
            error.append(count)
    elif ("Outcome: joy" in result):
        if (sample.label == 1):
            correct.append(count)
            print("joy")
        else:
            error.append(count)
    elif ("Outcome: love" in result):
        if (sample.label == 2):
            correct.append(count)
            print("love")
        else:
            error.append(count)
    elif ("Outcome: anger" in result):
        if (sample.label == 3):
            correct.append(count)
            print("anger")
        else:
            error.append(count)
    elif ("Outcome: fear" in result):
        if (sample.label == 4):
            correct.append(count)
            print("fear")
        else:
            error.append(count)
    elif ("Outcome: surprise" in result):
        if (sample.label == 5):
            correct.append(count)
            print("surprise")
        else:
            error.append(count)
    else: # error
        func_error.append(count)
        
    print(seconds, "seconds")

In [5]:
from tqdm import tqdm

for row in tqdm(new_df2.itertuples(index=False), total=new_df2.shape[0]):
    run_fhe_bert_tiny(row)
    count+=1

  0%|          | 0/60 [00:00<?, ?it/s]

sentence:  i feel so embarrassed


  2%|▏         | 1/60 [02:23<2:20:37, 143.01s/it]

[1.50501e+34, -4.35343e+33, 2.47357e+33, -4.23682e+33, 3.29983e+33, 5.26949e+33]
sadness
131.0 seconds
sentence:  i do feel stressed


  3%|▎         | 2/60 [04:49<2:19:59, 144.82s/it]

...

[-2.36487, -0.217259, 0.00398949, -2.59633, 0.244249, 6.66989]
surprise
183.0 seconds
sentence:  i am feeling amazing and seeing the difference


 98%|█████████▊| 59/60 [2:47:58<03:28, 208.38s/it]

[-1.88544, 3.14644, -1.34767, -2.5743, -1.29132, 3.76331]
surprise
184.0 seconds
sentence:  i always feel very shocked by that me threatening


100%|██████████| 60/60 [2:51:21<00:00, 171.37s/it]

[-1.80845, 4.0923, -0.54444, -1.66472, -0.624274, -0.995652]
191.0 seconds


In [6]:
len(correct)

37

In [7]:
len(error)

23

In [8]:
error

[2,
 8,
 19,
 21,
 22,
 23,
 24,
 25,
 27,
 28,
 29,
 32,
 33,
 35,
 40,
 43,
 46,
 48,
 50,
 51,
 54,
 55,
 59]

In [9]:
print(len(output))
print(len(execution_times))

60
60


In [10]:
import numpy as np
np.savetxt("output_labels2_60", output, delimiter=",")
np.savetxt("execution_time_labels2_60", output, delimiter=",")

Test again

In [11]:
import math

new_resamples = []
for i in range(len(output)):
    if (int(math.floor(math.log10(abs(output[i][0])))) > 0) or (int(math.floor(math.log10(abs(output[i][1])))) > 0) or (int(math.floor(math.log10(abs(output[i][2])))) > 0) or (int(math.floor(math.log10(abs(output[i][3])))) > 0) or (int(math.floor(math.log10(abs(output[i][4])))) > 0) or (int(math.floor(math.log10(abs(output[i][5])))) > 0):
        print(f"{i}: {output[i]}")
        new_resamples.append(i)
        
print(len(new_resamples))

0: [1.50501e+34, -4.35343e+33, 2.47357e+33, -4.23682e+33, 3.29983e+33, 5.26949e+33]
2: [-1.35031e+34, 6.65516e+33, -3.6084e+33, -3.41506e+33, -6.93341e+33, -1.52317e+33]
8: [-6.20319e+33, 8.09879e+33, 6.73397e+33, 1.00378e+33, -8.45743e+33, -5.26552e+33]
19: [2.67272e+32, -4.87804e+33, 9.53194e+33, 6.09975e+33, -9.5464e+33, 2.0382e+33]
24: [1396.81, 11753.3, -6349.39, -2375.95, -11437.0, 1732.91]
27: [-4.40172e+33, -4.96595e+33, -3.40025e+33, 1.29406e+34, -2.72487e+33, -2.3159e+34]
32: [-5.03405e+33, -9.12932e+33, -4.88635e+33, -7.44262e+32, -1.26616e+32, 7.514e+33]
33: [3000.23, 186.926, -2839.68, -10793.8, -1787.14, 4690.68]
35: [391.097, 2219.07, 5855.8, -4071.74, -1329.92, -5521.06]
40: [4.23163e+33, 3.10562e+33, 6.4249e+33, 8.30375e+33, 6.27592e+32, -4.69903e+33]
41: [8.34812e+33, -1.13859e+34, 4.37305e+33, 5.26754e+33, 1.05674e+34, 7.77129e+33]
43: [2.07569e+33, 4.71349e+32, 2.61528e+32, 1.81521e+33, 3.32125e+33, 1.27448e+34]
46: [-5.96194e+33, 5.08649e+33, -2.05645e+33, -1.69785

In [12]:
new_resamples

[0, 2, 8, 19, 24, 27, 32, 33, 35, 40, 41, 43, 46, 48, 50, 51, 54]

In [13]:
for i in new_resamples:
    if not i in error:
        print("not exist: ", i)
        new_resamples.remove(i)

not exist:  0
not exist:  41


In [14]:
correct2 = []
error2 = []
output2 = []
execution_times2 = []

func_error2 = []
count2 = 0

def run_fhe_bert_tiny2(sample):
    global correct2, error2, func_error2, output2, execution_times2, count2
    #print(sample)
    
    sentence = sample.text
    print("sentence: ", sentence)
    
    #cmd = [f"./FHE-BERT-Tiny", sentence]
    cmd = ["./FHE-BERT-Tiny", sentence, "--verbose"]
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )
    
    # format
    result = result.stdout.strip()
    #print("raw result", result)
    
    # check [x, x, x, x, x, x]
    #pattern = r'\[(.*?)\]'
    pattern = r'plain_result: \[(.*?)\]'
    match = re.search(pattern, result)
    tmp = match.group(1).split(",")
    res = [float(x) for x in tmp]
    output2.append(res)
    print(res)
    
    # execution time
    # The evaluation of the FHE circuit took: 121 seconds.
    pattern_time = r'The evaluation of the FHE circuit took:\s*(\d+)\s*seconds'
    match_time = re.search(pattern_time, result)
    seconds = float(match_time.group(1))
    execution_times2.append(seconds)
        
    if ("Outcome: sadness" in result):
        if (sample.label == 0):
            correct2.append(count2)
            print("sadness")
        else:
            error2.append(count2)
    elif ("Outcome: joy" in result):
        if (sample.label == 1):
            correct2.append(count2)
            print("joy")
        else:
            error2.append(count2)
    elif ("Outcome: love" in result):
        if (sample.label == 2):
            correct2.append(count2)
            print("love")
        else:
            error2.append(count)
    elif ("Outcome: anger" in result):
        if (sample.label == 3):
            correct2.append(count2)
            print("anger")
        else:
            error2.append(count)
    elif ("Outcome: fear" in result):
        if (sample.label == 4):
            correct2.append(count2)
            print("fear")
        else:
            error2.append(count2)
    elif ("Outcome: surprise" in result):
        if (sample.label == 5):
            correct2.append(count2)
            print("surprise")
        else:
            error2.append(count2)
    else: # error
        error2.append(count2)
        
    print(seconds, "seconds")

In [15]:
from tqdm import tqdm

eliminated = 0
for row in tqdm(new_df2.itertuples(index=False), total=new_df2.shape[0]):
    if count2 == new_resamples[eliminated]:
        #print(count)
        eliminated+=1
        run_fhe_bert_tiny2(row)
    count2+=1
    
    if eliminated == len(new_resamples):
        break

  0%|          | 0/60 [00:00<?, ?it/s]

sentence:  i feel so embarrassed


  2%|▏         | 1/60 [02:25<2:23:21, 145.80s/it]

[7.4698, -1.5028, -2.5878, -0.8139, -1.2505, -2.6787]
sadness
134.0 seconds
sentence:  i feels so lame


  5%|▌         | 3/60 [04:49<1:26:34, 91.14s/it] 

...

[-1.7937, 3.2962, -1.5136, -2.3728, -1.4786, 3.5385]
surprise
135.0 seconds
sentence:  i feel all funny sometimes


 87%|████████▋ | 52/60 [42:19<10:43, 80.42s/it]

[-1.389, -0.5553, 0.036, -1.9843, -0.6935, 6.3392]
surprise
146.0 seconds
sentence:  i feel shame in a strange way


 90%|█████████ | 54/60 [45:23<05:02, 50.44s/it]

[-1.5781590514270587e+34, -2.3408771184336498e+33, 3.0138184516333497e+33, 1.3403418492717354e+34, -1.9113091909456988e+33, 3.863590559895796e+33]
172.0 seconds


In [32]:
correct2.remove(0) # <- not need to run

In [33]:
len(correct2)

9

In [34]:
# total accuracy
total_accuracy = correct + correct2
total_accuracy.sort()
print(len(total_accuracy))

46


In [8]:
# 46 - 1 since forgot to remove the computed correct index (0 <- removed before, 41)
print(45/60)

0.75


In [35]:
len(total_accuracy)/60

0.7666666666666667

In [36]:
# Each label accuracy
label0_accuracy = []
label1_accuracy = []
label2_accuracy = []
label3_accuracy = []
label4_accuracy = []
label5_accuracy = []

for i in total_accuracy:
    if i < 10:
        label0_accuracy.append(i)
    elif i < 20:
        label1_accuracy.append(i)
    elif i < 30:
        label2_accuracy.append(i)
    elif i < 40:
        label3_accuracy.append(i)
    elif i < 50:
        label4_accuracy.append(i)
    elif i < 60:
        label5_accuracy.append(i)

In [37]:
print(len(label0_accuracy))
print(len(label1_accuracy))
print(len(label2_accuracy))
print(len(label3_accuracy))
print(len(label4_accuracy))
print(len(label5_accuracy))

10
10
3
9
7
7


In [21]:
print(df_label[df_label['label'] == 0].shape[0])
print(df_label[df_label['label'] == 1].shape[0])
print(df_label[df_label['label'] == 2].shape[0])
print(df_label[df_label['label'] == 3].shape[0])
print(df_label[df_label['label'] == 4].shape[0])
print(df_label[df_label['label'] == 5].shape[0])


581
695
159
275
224
66


In [44]:
new_resamples

[2, 8, 19, 24, 27, 32, 33, 35, 40, 43, 46, 48, 50, 51, 54]

In [46]:
# check final wrong answer
final_wrong = []

for i in range(60):
    if not i in total_accuracy and i in new_resamples:
        print(i)
        print(f"first: {output[i]}")
        print(f"retest: {output2[new_resamples.index(i)]}")
        final_wrong.append(i)

24
first: [1396.81, 11753.3, -6349.39, -2375.95, -11437.0, 1732.91]
retest: [-1.3212, 7.3302, -1.3484, -2.3303, -2.7216, -2.0867]
32
first: [-5.03405e+33, -9.12932e+33, -4.88635e+33, -7.44262e+32, -1.26616e+32, 7.514e+33]
retest: [-3.620196681219642e+33, -1.402486626341363e+34, 1.0423590575176966e+34, -7.78577360168551e+33, 6.129359245021199e+33, 1.006048827055768e+34]
43
first: [2.07569e+33, 4.71349e+32, 2.61528e+32, 1.81521e+33, 3.32125e+33, 1.27448e+34]
retest: [-1.2628, -2.2671, -2.2411, -0.999, 6.8369, -0.2977]
46
first: [-5.96194e+33, 5.08649e+33, -2.05645e+33, -1.69785e+34, -2.87309e+33, 1.30799e+33]
retest: [-2.5436198171556525e+34, -1.1266332631415636e+34, -1.225911328323534e+34, -1.0079157550837875e+34, -3.772004480638074e+33, -2.99968961278182e+33]
48
first: [-82255100.0, 52748100.0, 49897700.0, 77874500.0, -39277300.0, -48923300.0]
retest: [-5.427611663058691e+33, 6.533284008118717e+33, 8.309854844100619e+33, -1.9665765574514053e+34, -5.763807857270125e+33, 1.52609151465182

In [47]:
print(len(final_wrong))

6


In [6]:
import numpy as np
np.savetxt("output2_labels2_60", output, delimiter=",")
np.savetxt("execution_time2_labels2_60", output, delimiter=",")